# cap10_wf2020_coverage_readiness_v1

Purpose: audit whether the current cap10 joint-contract BC/checkpoint path can satisfy the retention handoff requirement to add `wf_2020` before actor retention.

This is a gate/readiness notebook, not a promotion artifact and not a training run. It uses existing cap10 sidecars plus the repo-owned walk-forward window definitions to identify whether the next full training loop can proceed without leaking evaluation labels.

In [1]:
from __future__ import annotations

import ast
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd())
CARD = ROOT / "runtime" / "model_selection" / "card" / "cap10_wf2020_coverage_readiness_v1"
CARD.mkdir(parents=True, exist_ok=True)
MODEL_SELECTION_CARD = ROOT / "runtime" / "model_selection" / "card"
REQUIRED_FOLDS = ["wf_2020", "wf_2022", "wf_2023_2024"]

SOURCES = {
    "workers": ROOT / "src" / "soxl_quant" / "app" / "workers.py",
    "retention_gate_summary": MODEL_SELECTION_CARD / "model_retention_walkforward_gate_v1" / "summary.json",
    "retention_gate_csv": MODEL_SELECTION_CARD / "model_retention_walkforward_gate_v1" / "retention_gate.csv",
    "retraining_handoff": MODEL_SELECTION_CARD / "retraining_gate_gap_handoff_v1" / "training_objective_handoff.json",
    "target_contract_summary": MODEL_SELECTION_CARD / "cap10_foldlocal_joint_target_contract_v1" / "summary.json",
    "target_contract_selection": MODEL_SELECTION_CARD / "cap10_foldlocal_joint_target_contract_v1" / "selection.json",
    "joint_target_contract": MODEL_SELECTION_CARD / "cap10_foldlocal_joint_target_contract_v1" / "joint_target_contract.csv",
    "joint_target_summary": MODEL_SELECTION_CARD / "cap10_foldlocal_joint_target_contract_v1" / "joint_target_summary.csv",
    "target_scope_audit": MODEL_SELECTION_CARD / "cap10_foldlocal_joint_target_contract_v1" / "target_scope_audit.csv",
    "bc_slice_summary": MODEL_SELECTION_CARD / "cap10_joint_contract_bc_slice_screen_v1" / "summary.json",
    "bc_fold_metrics": MODEL_SELECTION_CARD / "cap10_joint_contract_bc_slice_screen_v1" / "fold_bc_metrics.csv",
    "ppo_drift_coverage": MODEL_SELECTION_CARD / "cap10_joint_contract_bc_ppo_drift_screen_v1" / "coverage.csv",
    "ppo_fold_results": MODEL_SELECTION_CARD / "cap10_joint_contract_bc_ppo_drift_screen_v1" / "ppo_fold_results.csv",
    "checkpoint_selection": MODEL_SELECTION_CARD / "cap10_joint_contract_checkpoint_drift_selector_v1" / "checkpoint_selection_by_fold.csv",
    "checkpoint_candidates": MODEL_SELECTION_CARD / "cap10_joint_contract_checkpoint_drift_selector_v1" / "checkpoint_candidate_metrics.csv",
    "checkpoint_coverage": MODEL_SELECTION_CARD / "cap10_joint_contract_checkpoint_drift_selector_v1" / "coverage.csv",
    "checkpoint_selection_json": MODEL_SELECTION_CARD / "cap10_joint_contract_checkpoint_drift_selector_v1" / "selection.json",
    "sparse_release_coverage": MODEL_SELECTION_CARD / "sparse_release_bc_ppo_screen_v1" / "coverage.json",
    "sparse_release_coverage_csv": MODEL_SELECTION_CARD / "sparse_release_bc_ppo_screen_v1" / "coverage.csv",
}

ARTIFACTS = {
    "source_sidecar_audit": CARD / "source_sidecar_audit.csv",
    "wf2020_coverage_audit": CARD / "wf2020_coverage_audit.csv",
    "next_run_blockers": CARD / "next_run_blockers.csv",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}


def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path)

print({"card": rel(CARD), "sources": len(SOURCES)})

{'card': 'runtime/model_selection/card/cap10_wf2020_coverage_readiness_v1', 'sources': 19}


In [2]:
def load_walk_forward_windows(path: Path) -> pd.DataFrame:
    module = ast.parse(path.read_text(encoding="utf-8"))
    for node in module.body:
        if isinstance(node, ast.Assign):
            targets = node.targets
            value_node = node.value
        elif isinstance(node, ast.AnnAssign):
            targets = [node.target]
            value_node = node.value
        else:
            continue
        for target in targets:
            if isinstance(target, ast.Name) and target.id == "WALK_FORWARD_WINDOWS":
                value = ast.literal_eval(value_node)
                rows = []
                for name, train_seg, valid_seg, test_seg in value:
                    rows.append({
                        "fold": name,
                        "target_fold": f"{name}_train",
                        "train_start": train_seg[0],
                        "train_end": train_seg[1],
                        "valid_start": valid_seg[0],
                        "valid_end": valid_seg[1],
                        "test_start": test_seg[0],
                        "test_end": test_seg[1],
                    })
                return pd.DataFrame(rows)
    raise ValueError(f"WALK_FORWARD_WINDOWS not found in {path}")

wf_windows = load_walk_forward_windows(SOURCES["workers"])
wf_windows

,fold,target_fold,train_start,train_end,valid_start,valid_end,test_start,test_end
0,wf_2020,wf_2020_train,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31
1,wf_2022,wf_2022_train,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
2,wf_2023_2024,wf_2023_2024_train,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


In [3]:
source_rows = []
for name, path in SOURCES.items():
    row: dict[str, Any] = {
        "source": name,
        "path": rel(path),
        "exists": path.exists(),
        "rows": None,
        "columns": "",
        "fold_values": "",
        "has_wf_2020": False,
    }
    if path.exists() and path.suffix == ".csv":
        df = pd.read_csv(path)
        row["rows"] = len(df)
        row["columns"] = ",".join(map(str, df.columns))
        fold_values: list[str] = []
        for col in ["fold", "window", "target_fold", "source_target_fold"]:
            if col in df.columns:
                fold_values.extend(df[col].dropna().astype(str).unique().tolist())
        row["fold_values"] = ",".join(sorted(set(fold_values)))
        row["has_wf_2020"] = any("wf_2020" in value for value in fold_values)
    elif path.exists() and path.suffix == ".json":
        text = path.read_text(encoding="utf-8")
        payload = json.loads(text)
        row["rows"] = len(payload) if isinstance(payload, list) else len(payload.keys()) if isinstance(payload, dict) else None
        row["columns"] = ",".join(payload.keys()) if isinstance(payload, dict) else ""
        row["has_wf_2020"] = "wf_2020" in text
    elif path.exists() and path.suffix == ".py":
        row["has_wf_2020"] = "wf_2020" in path.read_text(encoding="utf-8")
    source_rows.append(row)

source_audit = pd.DataFrame(source_rows)
source_audit.to_csv(ARTIFACTS["source_sidecar_audit"], index=False)
source_audit

,source,path,exists,rows,columns,fold_values,has_wf_2020
0,workers,src/soxl_quant/app/workers.py,True,NaN,,,True
1,retention_gate_summary,runtime/model_selection/card/model_retention_w...,True,2.0,"rows,summary",,True
2,retention_gate_csv,runtime/model_selection/card/model_retention_w...,True,13.0,"profile,status,source_path,ytd_return_pct,ytd_...",,False
3,retraining_handoff,runtime/model_selection/card/retraining_gate_g...,True,17.0,"acceptance_criteria_for_retention,canonical_re...",,True
4,target_contract_summary,runtime/model_selection/card/cap10_foldlocal_j...,True,32.0,"acceptance_gate_for_next_run,bc_readiness_gate...",,True
5,target_contract_selection,runtime/model_selection/card/cap10_foldlocal_j...,True,30.0,"acceptance_gate_for_next_run,bc_readiness_gate...",,True
6,joint_target_contract,runtime/model_selection/card/cap10_foldlocal_j...,True,1261.0,"datetime,candidate,window,fold,target_scope,tr...","wf_2022_train,wf_2023_2024_train",False
7,joint_target_summary,runtime/model_selection/card/cap10_foldlocal_j...,True,2.0,"fold,rows,start,end,rebound_rows,downtrend_row...","wf_2022_train,wf_2023_2024_train",False
8,target_scope_audit,runtime/model_selection/card/cap10_foldlocal_j...,True,2.0,"window,target_scope,target_fold,fold,rows,trai...","wf_2022_train,wf_2023_2024_train",False
9,bc_slice_summary,runtime/model_selection/card/cap10_joint_contr...,True,27.0,"actor_retention,bc_screen_gate,canonical_basel...",,True


In [4]:
retention = load_json(SOURCES["retention_gate_summary"])
handoff = load_json(SOURCES["retraining_handoff"])
target_summary_json = load_json(SOURCES["target_contract_summary"])
checkpoint_selection_json = load_json(SOURCES["checkpoint_selection_json"])
sparse_release_coverage = load_json(SOURCES["sparse_release_coverage"])

joint_targets = load_csv(SOURCES["joint_target_contract"])
joint_target_summary = load_csv(SOURCES["joint_target_summary"])
target_scope = load_csv(SOURCES["target_scope_audit"])
bc_metrics = load_csv(SOURCES["bc_fold_metrics"])
ppo_coverage = load_csv(SOURCES["ppo_drift_coverage"])
checkpoint_coverage = load_csv(SOURCES["checkpoint_coverage"])
checkpoint_selection = load_csv(SOURCES["checkpoint_selection"])
checkpoint_candidates = load_csv(SOURCES["checkpoint_candidates"])

current_target_min_date = pd.to_datetime(joint_targets["datetime"]).min() if not joint_targets.empty else pd.NaT
current_target_max_date = pd.to_datetime(joint_targets["datetime"]).max() if not joint_targets.empty else pd.NaT
skipped = {
    item.get("fold"): item.get("reason")
    for item in sparse_release_coverage.get("skipped_windows", [])
    if isinstance(item, dict)
}

print({
    "current_target_min_date": None if pd.isna(current_target_min_date) else current_target_min_date.date().isoformat(),
    "current_target_max_date": None if pd.isna(current_target_max_date) else current_target_max_date.date().isoformat(),
    "skipped_folds": skipped,
    "handoff_required_controls": handoff.get("required_next_run_controls", []),
})

{'current_target_min_date': '2020-01-02', 'current_target_max_date': '2022-12-30', 'skipped_folds': {'wf_2020': 'sparse-release targets begin in 2020, so a wf_2020 warm-start would train on the test year'}, 'handoff_required_controls': ['add_wf_2020_coverage_before_actor_retention', 'use_bc_initialization_or_iter0_bc_equivalent_as_checkpoint_reference', 'apply_checkpoint_drift_stop: ppo_vs_bc_max_dd_delta_pp <= 2.0', 'apply_checkpoint_drift_stop: ppo_vs_bc_relative_delta_pp >= -5.0', 'require_fold_gate_before_full_training_continuation', 'export_daily_paths_for_dashboard_and_retention_gate_review']}


In [5]:
def count_rows(frame: pd.DataFrame, column: str, value: str) -> int:
    if frame.empty or column not in frame.columns:
        return 0
    return int(frame[column].astype(str).eq(value).sum())


def first_value(frame: pd.DataFrame, column: str, default: Any = None) -> Any:
    if frame.empty or column not in frame.columns:
        return default
    value = frame.iloc[0][column]
    return default if pd.isna(value) else value

coverage_rows: list[dict[str, Any]] = []
blockers: list[dict[str, Any]] = []
for wf in wf_windows.to_dict(orient="records"):
    fold = wf["fold"]
    target_fold = wf["target_fold"]
    target_rows = count_rows(joint_targets, "fold", target_fold)
    target_summary_rows = joint_target_summary[joint_target_summary.get("fold", pd.Series(dtype=str)).astype(str).eq(target_fold)].copy() if not joint_target_summary.empty else pd.DataFrame()
    scope_rows = target_scope[target_scope.get("fold", pd.Series(dtype=str)).astype(str).eq(target_fold)].copy() if not target_scope.empty else pd.DataFrame()
    bc_rows = bc_metrics[bc_metrics.get("fold", pd.Series(dtype=str)).astype(str).eq(target_fold)].copy() if not bc_metrics.empty else pd.DataFrame()
    ppo_cov_rows = ppo_coverage[ppo_coverage.get("fold", pd.Series(dtype=str)).astype(str).eq(fold)].copy() if not ppo_coverage.empty else pd.DataFrame()
    ckpt_cov_rows = checkpoint_coverage[checkpoint_coverage.get("fold", pd.Series(dtype=str)).astype(str).eq(fold)].copy() if not checkpoint_coverage.empty else pd.DataFrame()
    ckpt_selection_rows = checkpoint_selection[checkpoint_selection.get("fold", pd.Series(dtype=str)).astype(str).eq(fold)].copy() if not checkpoint_selection.empty else pd.DataFrame()
    ckpt_candidate_rows = checkpoint_candidates[checkpoint_candidates.get("fold", pd.Series(dtype=str)).astype(str).eq(fold)].copy() if not checkpoint_candidates.empty else pd.DataFrame()

    target_contract_present = target_rows > 0
    bc_metrics_present = not bc_rows.empty
    checkpoint_coverage_present = not ckpt_cov_rows.empty
    checkpoint_selection_present = not ckpt_selection_rows.empty
    candidate_metrics_present = not ckpt_candidate_rows.empty
    selected_candidate_gate = bool(first_value(ckpt_selection_rows, "selected_candidate_gate", False)) if checkpoint_selection_present else False
    selected_max_dd = first_value(ckpt_selection_rows, "selected_max_dd")
    selected_relative = first_value(ckpt_selection_rows, "selected_relative_return")
    non_leaking_current_source = target_contract_present and bool(first_value(scope_rows, "leakage_gate", False))
    if fold == "wf_2020" and not target_contract_present:
        non_leaking_current_source = False

    ready_for_actor_retention = all([
        target_contract_present,
        bc_metrics_present,
        checkpoint_coverage_present,
        checkpoint_selection_present,
        candidate_metrics_present,
        non_leaking_current_source,
        selected_candidate_gate,
    ])

    row = {
        **wf,
        "target_contract_present": target_contract_present,
        "target_contract_rows": target_rows,
        "target_contract_start": first_value(target_summary_rows, "start"),
        "target_contract_end": first_value(target_summary_rows, "end"),
        "scope_leakage_gate": first_value(scope_rows, "leakage_gate", False),
        "bc_metrics_present": bc_metrics_present,
        "bc_metric_rows": len(bc_rows),
        "ppo_coverage_present": not ppo_cov_rows.empty,
        "checkpoint_coverage_present": checkpoint_coverage_present,
        "checkpoint_selection_present": checkpoint_selection_present,
        "checkpoint_candidate_rows": len(ckpt_candidate_rows),
        "selected_candidate": first_value(ckpt_selection_rows, "selected_candidate"),
        "selected_candidate_gate": selected_candidate_gate,
        "selected_max_dd": selected_max_dd,
        "selected_relative_return": selected_relative,
        "current_source_skipped_reason": skipped.get(fold, ""),
        "non_leaking_current_source_available": non_leaking_current_source,
        "ready_for_actor_retention": ready_for_actor_retention,
    }
    coverage_rows.append(row)

    blocker_checks = {
        "missing_target_contract": not target_contract_present,
        "missing_bc_metrics": not bc_metrics_present,
        "missing_checkpoint_coverage": not checkpoint_coverage_present,
        "missing_checkpoint_selection": not checkpoint_selection_present,
        "missing_checkpoint_candidate_metrics": not candidate_metrics_present,
        "current_source_not_non_leaking": not non_leaking_current_source,
        "selected_candidate_gate_failed": checkpoint_selection_present and not selected_candidate_gate,
    }
    for blocker, active in blocker_checks.items():
        if active:
            blockers.append({
                "fold": fold,
                "target_fold": target_fold,
                "blocker": blocker,
                "detail": skipped.get(fold, "") if blocker == "current_source_not_non_leaking" else "",
            })

coverage_audit = pd.DataFrame(coverage_rows)
blocker_df = pd.DataFrame(blockers)
coverage_audit.to_csv(ARTIFACTS["wf2020_coverage_audit"], index=False)
blocker_df.to_csv(ARTIFACTS["next_run_blockers"], index=False)
coverage_audit

,fold,target_fold,train_start,train_end,valid_start,valid_end,test_start,test_end,target_contract_present,target_contract_rows,...,checkpoint_coverage_present,checkpoint_selection_present,checkpoint_candidate_rows,selected_candidate,selected_candidate_gate,selected_max_dd,selected_relative_return,current_source_skipped_reason,non_leaking_current_source_available,ready_for_actor_retention
0,wf_2020,wf_2020_train,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,False,0,...,False,False,0,None,False,NaN,NaN,"sparse-release targets begin in 2020, so a wf_...",False,False
1,wf_2022,wf_2022_train,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,True,505,...,True,True,4,bc_initialization,False,18.455088,36.419124,,True,False
2,wf_2023_2024,wf_2023_2024_train,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,True,756,...,True,True,4,bc_initialization,True,13.597475,30.501116,,True,True


In [6]:
retention_required = (handoff.get("retention_gate") or {}).get("required_wf_windows") or REQUIRED_FOLDS
missing_required = sorted(set(retention_required) - set(coverage_audit.loc[coverage_audit["ready_for_actor_retention"], "fold"]))
missing_target_folds = sorted(coverage_audit.loc[~coverage_audit["target_contract_present"], "fold"].tolist())
wf2020 = coverage_audit[coverage_audit["fold"].eq("wf_2020")].iloc[0].to_dict()

summary = {
    "experiment": "cap10_wf2020_coverage_readiness_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "wf2020_missing_target_contract_blocks_actor_retention",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_training_allowed": False,
    "selected_training_source": handoff.get("selected_training_source"),
    "required_wf_windows": retention_required,
    "ready_folds": sorted(coverage_audit.loc[coverage_audit["ready_for_actor_retention"], "fold"].tolist()),
    "missing_required_ready_folds": missing_required,
    "missing_target_contract_folds": missing_target_folds,
    "wf_2020": {
        "target_fold": wf2020["target_fold"],
        "train": [wf2020["train_start"], wf2020["train_end"]],
        "valid": [wf2020["valid_start"], wf2020["valid_end"]],
        "test": [wf2020["test_start"], wf2020["test_end"]],
        "target_contract_present": bool(wf2020["target_contract_present"]),
        "current_source_skipped_reason": wf2020["current_source_skipped_reason"],
        "current_target_min_date": None if pd.isna(current_target_min_date) else current_target_min_date.date().isoformat(),
        "non_leaking_current_source_available": bool(wf2020["non_leaking_current_source_available"]),
    },
    "controls_enforced": {
        "require_wf2020_before_actor_retention": "add_wf_2020_coverage_before_actor_retention" in handoff.get("required_next_run_controls", []),
        "checkpoint_drift_stop_required": any("checkpoint_drift_stop" in item for item in handoff.get("required_next_run_controls", [])),
        "retention_gate_max_dd_pct": (handoff.get("retention_gate") or {}).get("max_walkforward_drawdown_pct"),
    },
    "sidecars": {name: rel(path) for name, path in ARTIFACTS.items()},
    "next_action": "create_non_leaking_wf2020_foldlocal_target_contract_or_mark_cap10_joint_contract_ineligible_for_full_actor_retention",
}

ARTIFACTS["summary"].write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion_lines = [
    "# cap10_wf2020_coverage_readiness_v1",
    "",
    "- Decision: `wf2020_missing_target_contract_blocks_actor_retention`.",
    "- Current cap10 joint-contract path has target/BC/checkpoint evidence for `wf_2022` and `wf_2023_2024`, but not `wf_2020`.",
    f"- `wf_2020` worker split is train `{wf2020['train_start']} -> {wf2020['train_end']}`, valid `{wf2020['valid_start']} -> {wf2020['valid_end']}`, test `{wf2020['test_start']} -> {wf2020['test_end']}`.",
    f"- Current joint target contract begins at `{summary['wf_2020']['current_target_min_date']}` and the sparse-release source already records: `{summary['wf_2020']['current_source_skipped_reason']}`.",
    "- Actor retention and promotion remain blocked. Do not retain a cap10 actor from this branch until a non-leaking `wf_2020` target/BC/checkpoint sidecar exists or the branch is explicitly replaced.",
    "",
    "## Sidecars",
]
conclusion_lines.extend(f"- `{rel(path)}`" for path in ARTIFACTS.values())
ARTIFACTS["conclusion"].write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")
summary

{'experiment': 'cap10_wf2020_coverage_readiness_v1',
 'created_at': '2026-06-13T10:44:03.595220+00:00',
 'decision': 'wf2020_missing_target_contract_blocks_actor_retention',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'next_training_allowed': False,
 'selected_training_source': 'cap10_joint_contract_checkpoint_drift_selector_v1',
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'ready_folds': ['wf_2023_2024'],
 'missing_required_ready_folds': ['wf_2020', 'wf_2022'],
 'missing_target_contract_folds': ['wf_2020'],
 'wf_2020': {'target_fold': 'wf_2020_train',
  'train': ['2010-03-11', '2018-12-31'],
  'valid': ['2019-01-01', '2019-12-31'],
  'test': ['2020-01-01', '2020-12-31'],
  'target_contract_present': False,
  'current_source_skipped_reason': 'sparse-release targets begin in 2020, so a wf_2020 warm-start would train on the test year',
  'current_target_min_date': '2020-01-02',
  'non_leaking_current_source_available': False},
 'controls_enforced'

## Conclusion

This notebook intentionally blocks actor retention for the current cap10 joint-contract branch. The existing handoff requires `wf_2020`, `wf_2022`, and `wf_2023_2024` before retention. Current cap10 sidecars cover only `wf_2022` and `wf_2023_2024`, and the sparse-release source explicitly skipped `wf_2020` because the available target rows begin in the 2020 test year. The next valid step is a non-leaking `wf_2020` fold-local target contract or a replacement candidate path; continuing to retain an actor from the two-fold cap10 branch would violate the walk-forward gate.